In [28]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.datasets import load_iris

### PREPROCESAMIENTO

#### 1.- LIMPIEZA

In [29]:
def limpiar_dataset(df, target_col='Status'):
    
    df_clean = df.dropna()
    
    df_clean = df_clean.drop_duplicates()
    
    feature_cols = [col for col in df_clean.columns if col != target_col]
    df_clean = df_clean.groupby(feature_cols).filter(
        lambda grupo: grupo[target_col].nunique() == 1
    )
    
    return df_clean

#### 2.- CODIFICACIÓN

In [30]:
def encoding(df, target_col='Status'):
    df_num = df.copy()
    encoders = {}
    
    for col in df_num.columns:
        if df_num[col].dtype == 'object':
            try:
                df_num[col] = pd.to_numeric(df_num[col])
                continue  
            except ValueError:
                pass  
        
        if df_num[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df_num[col]):
            le = LabelEncoder()
            df_num[col] = le.fit_transform(df_num[col].astype(str))
            encoders[col] = le
        elif df_num[col].dtype == 'bool':
            df_num[col] = df_num[col].astype(int)
    return df_num, encoders

#### 3.- BINARIZACIÓN

In [31]:
def binarizar_data(df):

    df_proc = df.copy()
    metadata = {
        'columnas': list(df.columns),
        'minimos': {},
        'factores': {},
        'bits_por_columna': {}
    }
    
    bit_matrices = []  
    
    for col in df.columns:
        datos = df[col].values.astype(float)  
        
        # Eliminar negativos 
        min_val = np.min(datos)
        metadata['minimos'][col] = min_val
        if min_val < 0:
            datos = datos - min_val
        
        # Eliminar decimales 
        serie = pd.Series(datos)
        max_decimales = serie.astype(str).str.split('.').str[1].str.len().max()
        if pd.isna(max_decimales):
            max_decimales = 0
        
        if max_decimales > 0:
            factor = 10 ** max_decimales
            datos = datos * factor
            datos = np.round(datos).astype(np.int64)
        else:
            factor = 1
            datos = datos.astype(np.int64)
        metadata['factores'][col] = factor
        df_proc[col] = datos
        
        # Valor máximo y bits necesarios 
        max_val = int(datos.max())
        if max_val == 0:
            bits = 1
        else:
            bits = max_val.bit_length()
        metadata['bits_por_columna'][col] = bits
        
        # Conversión binaria vectorizada -----
        if bits > 0:
            shifts = np.arange(bits-1, -1, -1)  
            bits_matrix = ((datos[:, np.newaxis] >> shifts) & 1).astype(np.int8)
            bit_matrices.append(bits_matrix)
        else:
            bit_matrices.append(np.empty((len(datos), 0), dtype=np.int8))
    
    # Concatenar salida
    if bit_matrices:
        X_bin = np.hstack(bit_matrices)
    else:
        X_bin = np.empty((len(df), 0), dtype=np.int8)
    
    return X_bin, metadata

### OPERADORES

In [32]:
def operadores(x_val, y_val):
    if y_val == 1 and x_val == 1:
        return 1
    elif y_val == 1 and x_val == 0:
        return -1
    else:
        return 0

### GENERAR MATRIZ M

In [33]:
def genera_M(m, n):
    return np.zeros((m,n))

### ENTRENAMIENTO

In [34]:
def entrena(X, Y, m, n, muestras, matriz_M):
        
    for k in range(muestras):
        for j in range(m):
            for i in range(n):
                x_val = X[k, i]
                y_val = Y[k, j]
                matriz_M[j, i] += operadores(x_val, y_val)
    
    matriz_entrenada = pd.DataFrame(matriz_M)
    matriz_entrenada = matriz_entrenada.values
    return matriz_entrenada

### RECUPERACIÓN

In [35]:
def recuperacion(matriz_entrenada, x):
    resultados = matriz_entrenada @ x
    return np.argmax(resultados)

### VALIDACIÓN LOO

In [36]:
def leave_one_out_val(X, Y, m, n, funcion_entrena, funcion_recupera):

    num_patrones = X.shape[0]
    aciertos = 0
    resultados = []
    
    for i in range(num_patrones):
        X_train = np.delete(X, i, axis=0)
        Y_train = np.delete(Y, i, axis=0)
        
        muestras_train = X_train.shape[0]
        
        M = genera_M(m, n)
        
        matriz_entrenada = funcion_entrena(X_train, Y_train, m, n, muestras_train, M.copy())
        
        x_test = X[i].reshape(-1, 1) 
        clase_recuperada = funcion_recupera(matriz_entrenada, x_test)
        clase_esperada = i 
        
        es_correcto = (clase_recuperada == clase_esperada)
        aciertos += es_correcto
        resultados.append((i, es_correcto, clase_recuperada, clase_esperada))
        
        print(f"LOO: Dejando fuera patrón {i} -> Recuperado: {clase_recuperada}, Esperado: {clase_esperada}")
    
    precision = (aciertos / num_patrones) * 100
    return precision, resultados, aciertos

### Adquisición Iris

In [37]:
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['target'] = iris.target
print(f'Dimensiones datos: {df_iris.shape[0]}')
print(f'Dimensiones datos: {df_iris.shape[1]}')

Dimensiones datos: 150
Dimensiones datos: 5


### Preprocesar

In [42]:
df_limpio = limpiar_dataset(df_iris, target_col='target')
df_numerico, _ = encoding(df_limpio, target_col='target')

X_raw = df_numerico.drop('target', axis=1).values   
Y_raw = df_numerico['target'].values 

X, metadata_X = binarizar_data(pd.DataFrame(X_raw, columns=iris.feature_names))
Y = np.eye(3)[Y_raw]

print("Dimensiones de X_bin:", X.shape)  
print('------------------------------------------ Metadatos ------------------------------------------')
for llave, valor in metadata_X.items():
    print(f"{llave}: {valor}")
print('-----------------------------------------------------------------------------------------------')

m = Y.shape[1]
n = X.shape[1]
muestras = X.shape[0]

M_inicial = genera_M(m, n)

Dimensiones de X_bin: (149, 25)
------------------------------------------ Metadatos ------------------------------------------
columnas: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
minimos: {'sepal length (cm)': np.float64(4.3), 'sepal width (cm)': np.float64(2.0), 'petal length (cm)': np.float64(1.0), 'petal width (cm)': np.float64(0.1)}
factores: {'sepal length (cm)': 10, 'sepal width (cm)': 10, 'petal length (cm)': 10, 'petal width (cm)': 10}
bits_por_columna: {'sepal length (cm)': 7, 'sepal width (cm)': 6, 'petal length (cm)': 7, 'petal width (cm)': 5}
-----------------------------------------------------------------------------------------------


/tmp/ipykernel_105103/1431250841.py:13: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if df_num[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df_num[col]):


### Ejecutar LOO

In [39]:
# Ejecutar LOO 
precision_loo, resultados_loo, aciertos = leave_one_out_val(X, Y, m, n, entrena, recuperacion)
print("\n========== RESULTADO LEAVE-ONE-OUT ==========")
print(f"Precisión LOO: {precision_loo:.2f}% ({aciertos}/{X.shape[0]})")

LOO: Dejando fuera patrón 0 -> Recuperado: 0, Esperado: 0
LOO: Dejando fuera patrón 1 -> Recuperado: 1, Esperado: 1
LOO: Dejando fuera patrón 2 -> Recuperado: 0, Esperado: 2


LOO: Dejando fuera patrón 3 -> Recuperado: 1, Esperado: 3
LOO: Dejando fuera patrón 4 -> Recuperado: 0, Esperado: 4
LOO: Dejando fuera patrón 5 -> Recuperado: 0, Esperado: 5
LOO: Dejando fuera patrón 6 -> Recuperado: 0, Esperado: 6
LOO: Dejando fuera patrón 7 -> Recuperado: 0, Esperado: 7
LOO: Dejando fuera patrón 8 -> Recuperado: 1, Esperado: 8
LOO: Dejando fuera patrón 9 -> Recuperado: 1, Esperado: 9
LOO: Dejando fuera patrón 10 -> Recuperado: 0, Esperado: 10
LOO: Dejando fuera patrón 11 -> Recuperado: 0, Esperado: 11
LOO: Dejando fuera patrón 12 -> Recuperado: 1, Esperado: 12
LOO: Dejando fuera patrón 13 -> Recuperado: 1, Esperado: 13
LOO: Dejando fuera patrón 14 -> Recuperado: 0, Esperado: 14
LOO: Dejando fuera patrón 15 -> Recuperado: 1, Esperado: 15
LOO: Dejando fuera patrón 16 -> Recuperado: 0, Esperado: 16
LOO: Dejando fuera patrón 17 -> Recuperado: 0, Esperado: 17
LOO: Dejando fuera patrón 18 -> Recuperado: 0, Esperado: 18
LOO: Dejando fuera patrón 19 -> Recuperado: 0, Esperad